# 8. Review ranking and repair requests

Reproduce precision–recall from saved JEV probabilities and inspect human repair
requests by sentence role. Execution is offline. No generation or review service
is called. Install `python -m pip install -e '.[notebooks,review]'`.

A regeneration request is any independent human `redo` action. It does not need
a majority. Local `edit` actions are counted separately. Notebook 7 retains the
consensus agreement analysis; consensus does not define the human repair queue.

In [1]:
import json
from collections import Counter

from pandas import DataFrame

from fairforget.dataset_workflow import create_workspace, find_project
from fairforget.io import write_json
from fairforget.notebook_display import display
from fairforget.warbias.annotation_records import load_review_diagnostics
from fairforget.warbias.review_calibration import partition_rows
from fairforget.warbias.review_ranking import ranking_report
from fairforget.warbias.triplet_annotation import repair_targets, sentence_review_summary

project = find_project()
work = create_workspace("08_review_diagnostics")
scores, sentence_reviews, protocol, recorded = load_review_diagnostics(project)
targets = repair_targets(sentence_reviews)
split = json.loads((project / protocol["split"]).read_text())
development = partition_rows(targets, split, "development")
by_id = {row["id"]: row for row in scores}

## Human repair queue

Count each reviewer once per item. Keep undone and repeated reviews excluded
under the existing import rules. One `redo` queues an item even if other people
accept it. The [310-item list](../protocol/inputs/human_regeneration_requests.csv)
records the affected roles and number of redo votes.

Sentence counts overlap when more than one sentence was marked. Blank legacy
sentence reviews remain unknown. Generic or missing reason text is not assigned
a new issue type.

In [2]:
human_counts = {
    "items": len(targets),
    "regeneration_requests": sum(row["needs_regeneration"] for row in targets),
    "any_problem": sum(row["has_problem"] for row in targets),
    "local_edit_only": sum(
        row["has_local_edit"] and not row["needs_regeneration"] for row in targets
    ),
    "C_regeneration": sum(row["C_redo"] for row in targets),
}
display(human_counts)
patterns = Counter(
    ",".join(row["regeneration_roles"]) for row in targets if row["needs_regeneration"]
)
display(
    DataFrame([{"regeneration_roles": key, "triplets": count} for key, count in patterns.items()])
)
sentence_summary = sentence_review_summary(sentence_reviews)
display(sentence_summary)
DataFrame(targets).to_csv(work / "human-repair-targets.csv", index=False)
full_decisions = []
for version, references in recorded["full_current_decisions"].items():
    for reference, values in references.items():
        full_decisions.append({"version": version, "reference": reference, "items": 1059, **values})
display(DataFrame(full_decisions))

{'items': 1059,
 'regeneration_requests': 310,
 'any_problem': 351,
 'local_edit_only': 41,
 'C_regeneration': 290}

,regeneration_roles,triplets
0,S,13
1,"S,C,U",82
2,C,161
3,"S,C",35
4,"C,U",12
5,U,7


{'problem_review_rows': 533,
 'role_flag_counts': {'C': 472, 'S': 178, 'U': 134},
 'patterns': {'C': 324,
  'S,C': 42,
  'S': 33,
  'S,U': 3,
  'S,C,U': 100,
  'U': 25,
  'C,U': 6},
 'unique_items_by_role': {'S': 155, 'C': 319, 'U': 118},
 'counter_reason_counts': {'unnatural': 40,
  'grammar_or_spelling': 12,
  'counter_not_contradicting': 38,
  'non_parallel': 113,
  'unrelated_implies_stereotype': 5,
  'other': 6,
  'group_or_meaning': 12},
 'counter_reason_unknown_rows': 253}

,version,reference,items,f1,flagged,fn,fp,precision,recall,tn,tp
0,v1,any_problem,1059,0.319277,313,245,207,0.338658,0.301994,501,106
1,v1,redo,1059,0.311396,313,213,216,0.309904,0.312903,533,97
2,v2,any_problem,1059,0.213992,135,299,83,0.385185,0.148148,625,52
3,v2,redo,1059,0.202247,135,265,90,0.333333,0.145161,659,45


## Ranking whole-triplet repair requests

Use the maximum probability of a problem answer across each design's questions.
This is a ranking score, not a calibrated probability that the whole triplet has
a problem. Choice `confidence` is not used. Equal scores enter together.

The primary reference is any human redo request. The secondary reference also
includes local edits. Both use the same 846 development items. There are 236 redo
requests and 271 items with any human problem mark in this partition. No disputed
items are removed by majority voting.

Average precision uses the non-interpolated definition in the
[metric reference](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html).
Confidence intervals resample whole case families. Current decisions come from
actual saved choices; their polarity is normalized before comparison.

In [3]:
rankings = {}
for target in ["redo", "any_problem", "C_redo"]:
    reference = [
        {
            **row,
            "has_problem": row["needs_regeneration"]
            if target == "redo"
            else row["has_problem"]
            if target == "any_problem"
            else row[target],
        }
        for row in development
    ]
    variants = (
        ["atomic_C", "atomic_counter"] if target == "C_redo" else ["v1", "v2", "direct", "atomic"]
    )
    rankings[target] = {}
    for name in variants:
        predictions = [
            {
                **row,
                "score": by_id[row["id"]][name],
                "has_problem": by_id[row["id"]][name + "_flag"],
            }
            for row in development
        ]
        rankings[target][name] = ranking_report(
            predictions,
            reference,
            resamples=protocol["bootstrap"]["resamples"],
            seed=protocol["bootstrap"]["seed"],
        )
rows = []
for target in ["redo", "any_problem"]:
    for name, report in rankings[target].items():
        point = report["current_decision"]
        rows.append(
            {
                "reference": target,
                "variant": name,
                "items": report["compared_items"],
                "positives": report["positives"],
                "prevalence": report["prevalence"],
                "average_precision": report["average_precision"],
                "ap_ci95_low": report["average_precision_ci95"]["low"],
                "ap_ci95_high": report["average_precision_ci95"]["high"],
                "precision": point["precision"],
                "recall": point["recall"],
                "flagged": point["flagged"],
            }
        )
display(DataFrame(rows))

,reference,variant,items,positives,prevalence,average_precision,ap_ci95_low,ap_ci95_high,precision,recall,flagged
0,redo,v1,846,236,0.278960,0.317583,0.261809,0.388782,0.302920,0.351695,274
1,redo,v2,846,236,0.278960,0.324199,0.276568,0.385567,0.338983,0.169492,118
2,redo,direct,846,236,0.278960,0.278994,0.235147,0.337207,0.282383,0.461864,386
3,redo,atomic,846,236,0.278960,0.310500,0.257831,0.369553,0.285149,0.610169,505
4,any_problem,v1,846,271,0.320331,0.346987,0.292746,0.413244,0.332117,0.335793,274
5,any_problem,v2,846,271,0.320331,0.358523,0.309121,0.420058,0.381356,0.166052,118
6,any_problem,direct,846,271,0.320331,0.322356,0.273020,0.384278,0.318653,0.453875,386
7,any_problem,atomic,846,271,0.320331,0.340329,0.285216,0.402503,0.316832,0.590406,505


## Descriptive threshold trade-offs

These thresholds were evaluated on the same development labels. Their apparent
performance is optimistic and does not qualify a repair rule. The 80% recall
point illustrates workload; it is not an acceptance target or release setting.

In [4]:
operating_points = []
for name, report in rankings["redo"].items():
    for label, point in [
        ("best F1 on these labels", report["best_f1_descriptive"]),
        ("at least 80% recall", report["recall_targets_descriptive"]["0.8"]),
    ]:
        operating_points.append({"variant": name, "point": label, **point})
display(DataFrame(operating_points))

,variant,point,threshold,tp,fp,fn,flagged,precision,recall,f1
0,v1,best F1 on these labels,0.30,222,544,14,766,0.289817,0.940678,0.443114
1,v1,at least 80% recall,0.35,193,461,43,654,0.295107,0.817797,0.433708
2,v2,best F1 on these labels,0.19,215,503,21,718,0.299443,0.911017,0.450734
3,v2,at least 80% recall,0.22,190,462,46,652,0.291411,0.805085,0.427928
4,direct,best F1 on these labels,0.12,236,610,0,846,0.278960,1.000000,0.436229
5,direct,at least 80% recall,0.34,191,501,45,692,0.276012,0.809322,0.411638
6,atomic,best F1 on these labels,0.40,207,496,29,703,0.294452,0.877119,0.440895
7,atomic,at least 80% recall,0.43,193,459,43,652,0.296012,0.817797,0.434685


## Additional C review

C appears in 290 of 310 item-level redo requests. Only C is marked for redo in
161 items. Another 149 items include S or U requests. This supports extra C
review while keeping the full S/C/U check.

The saved atomic design already checks C grammar, naturalness, contradiction of
S and structure relative to S. Compare the maximum of these four problem scores
with C-specific human redo requests. Also show the contradiction question alone.
The target has 220 positive items among the 846 development items.

Human C marks include structure and language issues. They do not all mean that
C fails to contradict S. Missing typed reasons remain unknown.

In [5]:
display(
    DataFrame(
        [
            {
                "score": name,
                "items": report["compared_items"],
                "positives": report["positives"],
                "prevalence": report["prevalence"],
                "average_precision": report["average_precision"],
                "precision": report["current_decision"]["precision"],
                "recall": report["current_decision"]["recall"],
            }
            for name, report in rankings["C_redo"].items()
        ]
    )
)
write_json(work / "rankings.json", rankings)
result = {
    "status": "complete",
    "human_counts": human_counts,
    "development_items": len(development),
    "new_model_requests": False,
    "release_threshold_selected": False,
    "regenerated_items": 0,
}
write_json(work / "walkthrough-result.json", result)
display(result)

,score,items,positives,prevalence,average_precision,precision,recall
0,atomic_C,846,220,0.260047,0.278044,0.292308,0.259091
1,atomic_counter,846,220,0.260047,0.272135,0.333333,0.013636


{'status': 'complete',
 'human_counts': {'items': 1059,
  'regeneration_requests': 310,
  'any_problem': 351,
  'local_edit_only': 41,
  'C_regeneration': 290},
 'development_items': 846,
 'new_model_requests': False,
 'release_threshold_selected': False,
 'regenerated_items': 0}

## Repair workflow

All current synthetic triplets have a recorded first JEV review. A review flag
identifies a candidate issue; it is not enough to confirm an error automatically.
The saved ranking results do not qualify a threshold for automatic regeneration.

Keep a full-triplet check for every item and add focused C verification against
S and the taxonomy claim. Confirm the specific defect before repairing the
localized sentence. Preserve original text, review evidence and input hashes.
Review the complete repaired triplet again. Keep unresolved items pending; do
not silently mark them as passed. Human-reviewed and machine-reviewed versions
must retain separate provenance.